In [48]:
import pandas as pd
import re

## Importing Data

In [49]:
imports_df = pd.read_csv('../data/census_imports.csv')
exports_df = pd.read_csv('../data/census_exports.csv')
gdp_df = pd.read_csv('../data/gdp_industry.csv')

In [50]:
imports_df = imports_df[imports_df['time'].str.contains(r'(\d{4})-(12)')]
imports_df['YEAR'] = imports_df['time'].str.extract(r'(\d{4})-12')

exports_df = exports_df[exports_df['time'].str.contains(r'(\d{4})-(12)')]
exports_df['YEAR'] = exports_df['time'].str.extract(r'(\d{4})-12')

<positron-console-cell-50>:1: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
<positron-console-cell-50>:4: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.


In [51]:
imports_df = imports_df.drop(columns='time')
imports_df.columns = ['STATE','VALUE','NAICS','NAICS_LDESC','YEAR']

exports_df = exports_df.drop(columns='time')
exports_df.columns = ['STATE','VALUE','NAICS','NAICS_LDESC','YEAR']

## Cleaning and Transforming GDP by Capita

In [52]:
gdp_df = gdp_df.drop(columns=['Region','TableName','LineCode','Unit',
                     '1997','1998','1999','2000','2001','2002','2003','2004','2005','2006','2007','2008','2009'])

gdp_df = gdp_df.rename(columns={'GeoName':'REGION_NAME','IndustryClassification':'NAICS','Description':'NAICS_LDESC'})

In [53]:
state_abbr = { 'Alabama': 'AL', 'Alaska': 'AK', 'Arizona': 'AZ', 'Arkansas': 'AR', 
               'California': 'CA', 'Colorado': 'CO', 'Connecticut': 'CT', 
               'Delaware': 'DE', 'Florida': 'FL', 'Georgia': 'GA', 'Hawaii': 'HI', 
               'Idaho': 'ID', 'Illinois': 'IL', 'Indiana': 'IN', 'Iowa': 'IA', 'Kansas': 'KS', 'Kentucky': 'KY', 
               'Louisiana': 'LA', 'Maine': 'ME', 'Maryland': 'MD', 'Massachusetts': 'MA', 'Michigan': 'MI', 'Minnesota': 'MN', 'Mississippi': 'MS', 'Missouri': 'MO', 'Montana': 'MT', 
               'Nebraska': 'NE', 'Nevada': 'NV', 'New Hampshire': 'NH', 'New Jersey': 'NJ', 'New Mexico': 'NM', 'New York': 'NY', 'North Carolina': 'NC', 'North Dakota': 'ND', 
               'Ohio': 'OH', 'Oklahoma': 'OK', 'Oregon': 'OR', 'Pennsylvania': 'PA', 'Rhode Island': 'RI', 'South Carolina': 'SC', 'South Dakota': 'SD', 
               'Tennessee': 'TN', 'Texas': 'TX', 'Utah': 'UT', 'Vermont': 'VT', 'Virginia': 'VA', 'Washington': 'WA', 'West Virginia': 'WV', 'Wisconsin': 'WI', 'Wyoming': 'WY', 'United States': 'USA' }

In [54]:
gdp_df['REGION_NAME'] = gdp_df['REGION_NAME'].replace('United States *','United States')
gdp_df['STATE/REGION'] = gdp_df['REGION_NAME'].replace(state_abbr)

gdp_df['NAICS_LDESC'] = gdp_df['NAICS_LDESC'].str.strip()

In [55]:
gdp_df.head(5)

,GeoFIPS,REGION_NAME,NAICS,NAICS_LDESC,2010,2011,2012,2013,2014,2015,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,STATE/REGION
0,0,United States,...,All industry total,15048971,15599732,16253970,16880683,17608138,18295019,...,19612102,20656516,21539982.0,21375281,23725645,26054614,27811517,29298013,30762099,USA
1,0,United States,...,Private industries,12939469,13461114,14092708,14665532,15332504,15951002,...,17156255,18097765,18909779.0,18663991,20917717,23128892,24712793,26001672,27315089,USA
2,0,United States,11,"Agriculture, forestry, fishing and hunting",145743,179887,179457,215847,200581,182147,...,176840,177117,164214.0,164383,230723,294048,270441,269686,272014,USA
3,0,United States,111-112,Farms,117043,151135,148752,184472,167051,146259,...,138733,136825,122630.0,120506,185707,247070,218198,215830,(NA),USA
4,0,United States,113-115,"Forestry, fishing, and related activities",28700,28752,30705,31375,33530,35887,...,38107,40291,41585.0,43877,45016,46978,52243,53856,(NA),USA


In [56]:
gdp_df['NAICS'].unique()

<StringArray>
[                '...',                  '11',             '111-112',
             '113-115',                  '21',                 '211',
                 '212',                 '213',                  '22',
                  '23',               '31-33',         '321,327-339',
                 '321',                 '327',                 '331',
                 '332',                 '333',                 '334',
                 '335',           '3361-3363',      '3364-3466,3369',
                 '337',                 '339',     '311-316,322-326',
             '311-312',             '313-314',             '315-316',
                 '322',                 '323',                 '324',
                 '325',                 '326',                  '42',
               '44-45',               '48-49',                 '481',
                 '482',                 '483',                 '484',
                 '485',                 '486',         '487-488,492',
      

In [57]:
gdp_df['NAICS'] = gdp_df['NAICS'].replace('515517','515-517')

In [58]:
summary_desc = gdp_df.loc[gdp_df['NAICS'] == '...','NAICS_LDESC'].unique().tolist()
summ_codemap = {desc: f'..{i}' for i, desc in enumerate(summary_desc)}

In [59]:
gdp_df.loc[gdp_df['NAICS'].str.contains(r'\.\.',regex=True), 'NAICS'] = gdp_df.loc[gdp_df['NAICS'].str.contains(r'\.\.',regex=True), 'NAICS_LDESC'].map(summ_codemap)

In [60]:
gdp_df.loc[gdp_df['NAICS'].str.contains(',')]['NAICS'].unique()

<StringArray>
[        '321,327-339',      '3364-3466,3369',     '311-316,322-326',
         '487-488,492',               '52,53',            '54,55,56',
 '5412-5414,5416-5419',               '61,62',               '71,72',
               '11,21',            '42,44-45',            '22,48-49',
            '31-33,51']
Length: 13, dtype: str

In [61]:
# Variables to identify whether aggregate codes are exclusive (groups without other given rows) 
#                                               or inclusive (groupings from BEA)
excl_groups = ['3364-3466,3369','487-488,492','5412-5414,5416-5419']
incl_groups = ['311-316,322-326','321,327-339','52,53','54,55,56','61,62','71,72','11,21','42,44-45','22,48-49','31-33,51']

In [62]:
gdp_codes = gdp_df['NAICS'].unique().tolist()

In [63]:
gdp_codes

['..0',
 '..1',
 '11',
 '111-112',
 '113-115',
 '21',
 '211',
 '212',
 '213',
 '22',
 '23',
 '31-33',
 '321,327-339',
 '321',
 '327',
 '331',
 '332',
 '333',
 '334',
 '335',
 '3361-3363',
 '3364-3466,3369',
 '337',
 '339',
 '311-316,322-326',
 '311-312',
 '313-314',
 '315-316',
 '322',
 '323',
 '324',
 '325',
 '326',
 '42',
 '44-45',
 '48-49',
 '481',
 '482',
 '483',
 '484',
 '485',
 '486',
 '487-488,492',
 '493',
 '51',
 '511',
 '512',
 '515-517',
 '518-519',
 '52,53',
 '52',
 '521-522',
 '523',
 '524',
 '525',
 '53',
 '531',
 '532-533',
 '54,55,56',
 '54',
 '5411',
 '5415',
 '5412-5414,5416-5419',
 '55',
 '56',
 '561',
 '562',
 '61,62',
 '61',
 '62',
 '621',
 '622',
 '623',
 '624',
 '71,72',
 '71',
 '711-712',
 '713',
 '72',
 '721',
 '722',
 '81',
 '92',
 '..2',
 '..3',
 '..4',
 '11,21',
 '42,44-45',
 '22,48-49',
 '31-33,51',
 '..5',
 '..6',
 '..7',
 '..8',
 '..9',
 '..10']

In [64]:
code_map = {}
level_map = {}
for code in gdp_codes:
    if ('-' in code) & (code not in incl_groups):
        if ',' in code:
            first, second = re.search(r'(\d+),(\d+)', code).groups()
            if '-' in first:
                start, end = re.search(r'(\d+)-(\d+)',first).groups()
                for i in range(int(start), int(end)+1):
                    code_map[str(i).ljust(4,'0')] = code

                level_map[code] = len(str(start))
            else:
                code_map[first.ljust(4,'0')] = code

                level_map[code] = len(str(first))

            if '-' in second:
                start, end = re.search(r'(\d+)-(\d+)',second).groups()
                for i in range(int(start), int(end)+1):
                    code_map[str(i).ljust(4,'0')] = code
            else:
                code_map[second.ljust(4,'0')] = code
        elif '-' in code:
            start, end = re.search(r'(\d+)-(\d+)', code).groups()
            for i in range(int(start), int(end)+1):
                code_map[str(i).ljust(4,'0')] = code

            level_map[code] = len(str(start))
    elif (code not in incl_groups) and not re.search(r'^\.\.', code):
        code_map[code.ljust(4,'0')] = code

        level_map[code] = len(code)
    else:
        level_map[code] = 1

In [65]:
code_map

{'1100': '11',
 '1110': '111-112',
 '1120': '111-112',
 '1130': '113-115',
 '1140': '113-115',
 '1150': '113-115',
 '2100': '21',
 '2110': '211',
 '2120': '212',
 '2130': '213',
 '2200': '22',
 '2300': '23',
 '3100': '31-33',
 '3200': '31-33',
 '3300': '31-33',
 '3210': '321',
 '3270': '327',
 '3310': '331',
 '3320': '332',
 '3330': '333',
 '3340': '334',
 '3350': '335',
 '3361': '3361-3363',
 '3362': '3361-3363',
 '3363': '3361-3363',
 '3466': '3364-3466,3369',
 '3369': '3364-3466,3369',
 '3370': '337',
 '3390': '339',
 '3110': '311-312',
 '3120': '311-312',
 '3130': '313-314',
 '3140': '313-314',
 '3150': '315-316',
 '3160': '315-316',
 '3220': '322',
 '3230': '323',
 '3240': '324',
 '3250': '325',
 '3260': '326',
 '4200': '42',
 '4400': '44-45',
 '4500': '44-45',
 '4800': '48-49',
 '4900': '48-49',
 '4810': '481',
 '4820': '482',
 '4830': '483',
 '4840': '484',
 '4850': '485',
 '4860': '486',
 '4880': '487-488,492',
 '4920': '487-488,492',
 '4930': '493',
 '5100': '51',
 '5110': '51

In [66]:
gdp_df['LEVEL'] = gdp_df['NAICS'].replace(level_map)
gdp_df.loc[gdp_df['NAICS'].isin(incl_groups), 'LEVEL'] = 0

In [67]:
gdp_df_l = pd.melt(gdp_df,
                   id_vars=['REGION_NAME','STATE/REGION','NAICS','NAICS_LDESC','LEVEL'],
                    value_vars=['2010','2011','2012','2013','2014','2015',
                                '2016','2017','2018','2019','2020','2021','2022','2023','2024','2025'],
                    var_name='YEAR',
                    value_name='GDP')

gdp_df_l['GDP'] = pd.to_numeric(gdp_df_l['GDP'], errors='coerce') * 1000000

In [68]:
region_list = ['New England', 'Mideast', 'Great Lakes', 'Plains', 
               'Southeast', 'Southwest', 'Rocky Mountain', 'Far West']

gdp_states = gdp_df_l[~gdp_df_l['STATE/REGION'].isin(region_list)]

In [69]:
gdp_states

,REGION_NAME,STATE/REGION,NAICS,NAICS_LDESC,LEVEL,YEAR,GDP
0,United States,USA,..0,All industry total,1,2010,1.504897e+13
1,United States,USA,..1,Private industries,1,2010,1.293947e+13
2,United States,USA,11,"Agriculture, forestry, fishing and hunting",2,2010,1.457430e+11
3,United States,USA,111-112,Farms,3,2010,1.170430e+11
4,United States,USA,113-115,"Forestry, fishing, and related activities",3,2010,2.870000e+10
...,...,...,...,...,...,...,...
87643,Wyoming,WY,"42,44-45",Trade,0,2025,5.349000e+09
87644,Wyoming,WY,"22,48-49",Transportation and utilities,0,2025,5.444100e+09
87645,Wyoming,WY,"31-33,51",Manufacturing and information,0,2025,NaN
87646,Wyoming,WY,..5,Private goods-producing industries 2/,1,2025,NaN


## Loading Census Population Data

In [70]:
pop_df = pd.read_csv('../data/population_est.csv')

In [71]:
pop_df = pd.melt(pop_df,
    id_vars=['STATE'],
    value_vars=['2010','2011','2012','2013','2014','2015','2016','2017',
               '2018','2019','2020','2021','2022','2023','2024','2025'],
    var_name='YEAR',
    value_name='EST_POP')

In [72]:
pop_df['STATE/REGION'] = pop_df['STATE'].replace('United States','USA').replace(state_abbr)
pop_df = pop_df.drop(columns=['STATE'])

In [73]:
pop_df

,YEAR,EST_POP,STATE/REGION
0,2010,4785514,AL
1,2010,713982,AK
2,2010,6407342,AZ
3,2010,2921998,AR
4,2010,37319550,CA
...,...,...,...
827,2025,8001020,WA
828,2025,1766147,WV
829,2025,5972787,WI
830,2025,588753,WY


## Cleaning and Transforming Census Imports

In [74]:
imports_df.head(5)

,STATE,VALUE,NAICS,NAICS_LDESC,YEAR
92722,-,333889708,1122,SWINE,2013
92723,-,333889708,11221,SWINE,2013
92724,-,333889708,112210,SWINE,2013
92725,-,56965631,1123,POULTRY AND EGGS,2013
92726,-,56965631,1123X,POULTRY AND EGGS,2013


In [75]:
imports_df['STATE'] = imports_df['STATE'].replace('-','USA')

In [76]:
imports_df['NAICS'].unique()

<StringArray>
[  '1122',  '11221', '112210',   '1123',  '1123X', '1123XX',   '1124',
  '11241', '112410',  '11242',
 ...
 '11211X',    '115',   '1151',   '1152',     '51',    '511',   '5112',
     '92',    '920',   '9200']
Length: 781, dtype: str

In [77]:
imports_df['NAICS'] = imports_df['NAICS'].str.replace('0','').str.replace('X','')

imports_df['NAICS_pad'] = imports_df['NAICS'].str[:4].str.ljust(4,'0')
import_codes = imports_df['NAICS_pad'].unique().tolist()

imports_df = imports_df.drop_duplicates(subset=['STATE','VALUE','YEAR','NAICS'])

In [78]:
imports_df['LEVEL'] = imports_df['NAICS'].str.len()

imports_df.loc[imports_df['NAICS'] == '31-33', 'LEVEL'] = 2

In [79]:
for code in import_codes:
    if code == '-000':
        code_map[code] = '..0'
    elif code == '31-3':
        code_map[code] = '31-33'
    elif code not in code_map.keys():
        code_map[code] = 'NA'
        for i in range(4):
            pad_code = code[:-1] + '0' * (i + 1)
            if pad_code in code_map.keys():
                code_map[code] = code_map[pad_code]
                break

In [80]:
imports_df['GDP_CODE'] = imports_df['NAICS_pad'].replace(code_map)

In [81]:
imports_df = imports_df.drop(columns=['NAICS_pad']).rename(columns={
    'STATE':'STATE/REGION',
    'VALUE':'IMPORT_VAL',
    'NAICS':'IMP_NAICS',
    'NAICS_LDESC':'IMP_NAICS_DESC',
    'LEVEL':'IMP_LEVEL'})

In [82]:
imports_df

,STATE/REGION,IMPORT_VAL,IMP_NAICS,IMP_NAICS_DESC,YEAR,IMP_LEVEL,GDP_CODE
92722,USA,333889708,1122,SWINE,2013,4,111-112
92723,USA,333889708,11221,SWINE,2013,5,111-112
92725,USA,56965631,1123,POULTRY AND EGGS,2013,4,111-112
92728,USA,14396070,1124,"SHEEP, GOATS AND FINE ANIMAL HAIR",2013,4,111-112
92729,USA,10686925,11241,SHEEP AND WOOL,2013,5,111-112
...,...,...,...,...,...,...,...
1534263,WA,315932005,99,OTHER SPECIAL CLASSIFICATION PROVISIONS,2025,2,NA
1534264,WI,239760200,99,OTHER SPECIAL CLASSIFICATION PROVISIONS,2025,2,NA
1534265,WV,8890626,99,OTHER SPECIAL CLASSIFICATION PROVISIONS,2025,2,NA
1534266,WY,15238401,99,OTHER SPECIAL CLASSIFICATION PROVISIONS,2025,2,NA


## Cleaning and Transforming Census Exports

In [83]:
exports_df.head(5)

,STATE,VALUE,NAICS,NAICS_LDESC,YEAR
92265,-,79737309998,11,AGRICULTURE AND LIVESTOCK PRODUCTS,2013
92266,-,68935141909,111,AGRICULTURAL PRODUCTS,2013
92267,-,41576840935,1111,OILSEEDS AND GRAINS,2013
92268,-,21605835043,11111,SOYBEANS,2013
92269,-,21605835043,111110,SOYBEANS,2013


In [84]:
exports_df['STATE'] = exports_df['STATE'].replace('-','USA')

In [85]:
exports_df['NAICS'].unique()

<StringArray>
[    '11',    '111',   '1111',  '11111', '111110',  '11112',  '11114',
 '111120',  '11113', '111130',
 ...
 '327211', '327212',    '115',   '1152',     '51',    '511',   '5112',
     '92',    '920',   '9200']
Length: 775, dtype: str

In [86]:
exports_df['NAICS'] = exports_df['NAICS'].str.replace('0','').str.replace('X','')

exports_df['NAICS_pad'] = exports_df['NAICS'].str[:4].str.ljust(4,'0')
export_codes = exports_df['NAICS_pad'].unique().tolist()

exports_df = exports_df.drop_duplicates(subset=['STATE','VALUE','YEAR','NAICS'])

In [87]:
exports_df['LEVEL'] = exports_df['NAICS'].str.len()

exports_df.loc[exports_df['NAICS'] == '31-33', 'LEVEL'] = 2

In [88]:
for code in export_codes:
    if code == '-000':
        code_map[code] = '..0'
    elif code == '31-3':
        code_map[code] = '31-33'
    elif code not in code_map.keys():
        code_map[code] = 'NA'
        for i in range(4):
            pad_code = code[:-1] + '0' * (i + 1)
            if pad_code in code_map.keys():
                code_map[code] = code_map[pad_code]
                break

In [89]:
exports_df['GDP_CODE'] = exports_df['NAICS_pad'].replace(code_map)

In [90]:
exports_df = exports_df.drop(columns=['NAICS_pad']).rename(columns={
    'STATE':'STATE/REGION',
    'VALUE':'EXPORT_VAL',
    'NAICS':'EXP_NAICS',
    'NAICS_LDESC':'EXP_NAICS_DESC',
    'LEVEL':'EXP_LEVEL'})

In [91]:
exports_df

,STATE/REGION,EXPORT_VAL,EXP_NAICS,EXP_NAICS_DESC,YEAR,EXP_LEVEL,GDP_CODE
92265,USA,79737309998,11,AGRICULTURE AND LIVESTOCK PRODUCTS,2013,2,11
92266,USA,68935141909,111,AGRICULTURAL PRODUCTS,2013,3,111-112
92267,USA,41576840935,1111,OILSEEDS AND GRAINS,2013,4,111-112
92268,USA,21605835043,11111,SOYBEANS,2013,5,111-112
92270,USA,458793743,11112,OILSEEDS (EXCEPT SOYBEAN),2013,5,111-112
...,...,...,...,...,...,...,...
1511797,XX,535298644,3322,CUTLERY AND HANDTOOLS,2025,4,332
1511798,XX,165509591,3323,ARCHITECTURAL AND STRUCTURAL METALS,2025,4,332
1511799,XX,122015935,3324,"BOILERS, TANKS, AND SHIPPING CONTAINERS",2025,4,332
1511800,XX,546389952,3325,HARDWARE,2025,4,332


## Combining & Exporting Datasets

In [92]:
gdp_states = gdp_states.merge(
    pop_df,
    on=['STATE/REGION','YEAR'],
    how='left'
)

In [93]:
exports_df

,STATE/REGION,EXPORT_VAL,EXP_NAICS,EXP_NAICS_DESC,YEAR,EXP_LEVEL,GDP_CODE
92265,USA,79737309998,11,AGRICULTURE AND LIVESTOCK PRODUCTS,2013,2,11
92266,USA,68935141909,111,AGRICULTURAL PRODUCTS,2013,3,111-112
92267,USA,41576840935,1111,OILSEEDS AND GRAINS,2013,4,111-112
92268,USA,21605835043,11111,SOYBEANS,2013,5,111-112
92270,USA,458793743,11112,OILSEEDS (EXCEPT SOYBEAN),2013,5,111-112
...,...,...,...,...,...,...,...
1511797,XX,535298644,3322,CUTLERY AND HANDTOOLS,2025,4,332
1511798,XX,165509591,3323,ARCHITECTURAL AND STRUCTURAL METALS,2025,4,332
1511799,XX,122015935,3324,"BOILERS, TANKS, AND SHIPPING CONTAINERS",2025,4,332
1511800,XX,546389952,3325,HARDWARE,2025,4,332


In [94]:
# Dropping summary rows for more accurate aggregations in dashboard (may exclude GDP value that cannot be tracked to individual states)
gdp_states = gdp_states[gdp_states['STATE/REGION'] != 'USA']

imports_df = imports_df[(imports_df['STATE/REGION'] != 'USA') & (imports_df['STATE/REGION'] != 'XX')]
exports_df = exports_df[(exports_df['STATE/REGION'] != 'USA') & (exports_df['STATE/REGION'] != 'XX')]

gdp_states = gdp_states[gdp_states['YEAR'] != '2025']
imports_df = imports_df[imports_df['YEAR'] != '2025']
exports_df = exports_df[exports_df['YEAR'] != '2025']

In [95]:
imports_df.to_csv('../outputs/census_imports.csv',index=False)
exports_df.to_csv('../outputs/census_exports.csv',index=False)

gdp_states.to_csv('../outputs/gdp_states.csv',index=False)

## Creating Summary Tables

In [96]:
import_join = (imports_df.groupby(['GDP_CODE','YEAR','STATE/REGION'])['IMPORT_VAL']
    .sum()
    .reset_index()
    .rename(columns={'IMPORT_VAL':'GDP_IMP_SUM'}))
export_join = (exports_df.groupby(['GDP_CODE','YEAR','STATE/REGION'])['EXPORT_VAL']
    .sum()
    .reset_index()
    .rename(columns={'EXPORT_VAL':'GDP_EXP_SUM'}))

In [97]:
gdp_imports = import_join.merge(
    gdp_states, 
    left_on=['STATE/REGION','YEAR','GDP_CODE'],
    right_on=['STATE/REGION','YEAR','NAICS'],
    how='inner').drop(columns=['GDP_CODE']).merge(
    pop_df,
    on=['STATE/REGION','YEAR'],
    how='inner'
    )
gdp_exports = export_join.merge(
    gdp_states,
    left_on=['STATE/REGION','YEAR','GDP_CODE'],
    right_on=['STATE/REGION','YEAR','NAICS'],
    how='inner').drop(columns=['GDP_CODE']).merge(
    pop_df,
    on=['STATE/REGION','YEAR'],
    how='inner'
    )

gdp_imports.to_csv('../outputs/gdp_imports.csv',index=False)
gdp_exports.to_csv('../outputs/gdp_exports.csv',index=False)